# SoleSearch AI — Step 4: Build the colorway gallery automatically

This downloads the public **"50k+ Sneaker Images"** dataset on Kaggle **inside Colab** (about 9 GB, nothing
touches your laptop). Its folders are named after StockX listings, which include the colorway, for example
*"Air Jordan 1 Retro High Chicago"*. The notebook finds the folders for each of your colorways, takes up to 5
photos each, shrinks them to 600 px, and gives you one small `gallery.zip` (a few MB) to upload to GitHub.

The dataset is from around 2019, so newer colorways (like Samba OG or Dunk Low Panda re-releases) may be
missing. The notebook tells you which ones it couldn't find. No GPU needed.

## 1. Download the dataset (5–10 minutes)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    pass  # fine for public datasets

import kagglehub
from pathlib import Path
DS = Path(kagglehub.dataset_download('sebastiaanjohn/sneakers'))
folders = [d for d in DS.rglob('*') if d.is_dir() and any(f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'} for f in d.iterdir())]
print(len(folders), 'sneaker folders found')

## 2. Match folders to your colorways

Each colorway has words that must appear in the folder name and words that rule it out (so "Bred Toe" doesn't
count as "Bred", and SB Dunks don't count as Dunk Lows). **Read the printout**: if a match looks wrong, add a
word to that colorway's exclude list and re-run. To add a colorway, add a line in the same format.

In [ ]:
import re
def norm(s): return re.sub(r'[^a-z0-9]+', ' ', s.lower()).strip()
# our colorway folder -> (regexes that must ALL match, regexes that must NOT match)
WANT = {
 ('air_jordan_1', 'chicago'):      ([r'\bjordan 1\b', r'\bchicago\b'], [r'\blow\b', r'\bmid\b', r'\blost\b']),
 ('air_jordan_1', 'bred'):         ([r'\bjordan 1\b', r'\bbred\b'], [r'\blow\b', r'\bmid\b', r'\btoe\b', r'\bpatent\b']),
 ('air_jordan_1', 'royal'):        ([r'\bjordan 1\b', r'\broyal\b'], [r'\blow\b', r'\bmid\b', r'\btoe\b']),
 ('air_jordan_4', 'white_cement'): ([r'\bjordan 4\b', r'\bwhite cement\b'], []),
 ('air_jordan_4', 'military_black'): ([r'\bjordan 4\b', r'\bmilitary black\b'], []),
 ('air_jordan_4', 'black_cat'):    ([r'\bjordan 4\b', r'\bblack cat\b'], []),
 ('nike_dunk_low', 'panda'):       ([r'\bdunk low\b', r'\bwhite black\b|\bpanda\b'], [r'\bsb\b', r'\bgs\b']),
 ('nike_dunk_low', 'grey_fog'):    ([r'\bdunk low\b', r'\bgrey fog\b'], [r'\bsb\b']),
 ('nike_air_force_1', 'triple_white'): ([r'\bair force 1\b', r'\bwhite\b'], [r'\bblack\b', r'\bred\b', r'\bblue\b', r'\bmid\b', r'\bhigh\b', r'\boff white\b', r'\bgreen\b', r'\bgum\b']),
 ('nike_air_force_1', 'triple_black'): ([r'\bair force 1\b', r'\bblack\b'], [r'\bwhite\b', r'\bmid\b', r'\bhigh\b', r'\bred\b', r'\bgum\b']),
 ('yeezy_boost_350', 'zebra'):     ([r'\b350\b', r'\bzebra\b'], []),
 ('yeezy_boost_350', 'bred'):      ([r'\b350\b', r'\bbred\b|\bblack red\b'], []),
 ('yeezy_boost_350', 'onyx'):      ([r'\b350\b', r'\bonyx\b'], []),
 ('adidas_samba', 'white_black'):  ([r'\bsamba\b', r'\bwhite\b.*\bblack\b'], [r'\bwales\b']),
 ('adidas_samba', 'black_white'):  ([r'\bsamba\b', r'\bblack\b.*\bwhite\b'], [r'\bwales\b']),
 ('new_balance_550', 'white_green'): ([r'\b550\b', r'\bwhite\b', r'\bgreen\b'], []),
 ('new_balance_550', 'white_grey'):  ([r'\b550\b', r'\bwhite\b', r'\bgr[ae]y\b'], []),
}
def matches(folder, inc, exc):
    n = norm(folder)
    return all(re.search(p, n) for p in inc) and not any(re.search(p, n) for p in exc)

found = {}
for key, (inc, exc) in WANT.items():
    found[key] = [d for d in folders if matches(d.name, inc, exc)]
    names = ', '.join(d.name for d in found[key][:4]) + (' ...' if len(found[key]) > 4 else '')
    print(f'{key[0]}/{key[1]:15s} {len(found[key]):3d} folders  {names or "(none found)"}')

## 3. Shrink up to 5 photos per colorway and download `gallery.zip`

In [ ]:
import shutil
from PIL import Image, ImageOps

PER_COLORWAY = 5
OUT = Path('/content/gallery')
shutil.rmtree(OUT, ignore_errors=True)
IMG = {'.jpg', '.jpeg', '.png', '.webp'}

total_files, total_bytes, missing = 0, 0, []
for (model, colorway), dirs in found.items():
    # take photos round-robin across matching folders, for variety
    pools = [sorted(f for f in d.iterdir() if f.suffix.lower() in IMG) for d in dirs]
    picked, i = [], 0
    while len(picked) < PER_COLORWAY and any(i < len(p) for p in pools):
        picked += [p[i] for p in pools if i < len(p)]
        i += 1
    saved = 0
    for src in picked:
        if saved >= PER_COLORWAY:
            break
        try:
            img = ImageOps.exif_transpose(Image.open(src)).convert('RGB')
            img.thumbnail((600, 600))
            out = OUT / model / colorway / f'{colorway}_{saved + 1}.jpg'
            out.parent.mkdir(parents=True, exist_ok=True)
            img.save(out, 'JPEG', quality=82)
            saved += 1; total_files += 1; total_bytes += out.stat().st_size
        except Exception:
            continue  # skip broken images
    if saved == 0:
        missing.append(f'{model}/{colorway}')

print(f'{total_files} photos, {total_bytes / 1e6:.1f} MB total')
if missing:
    print('Not found (add your own photos for these, or remove them):', ', '.join(missing))

## 4. Check them before downloading

Mislabeled photos happen in scraped datasets. If one looks wrong, delete it in the Colab file browser
(folder icon on the left, under `gallery/`) and re-run the last cell of this step.

In [ ]:
import matplotlib.pyplot as plt
for d in sorted(OUT.glob('*/*')):
    files = sorted(d.glob('*.jpg'))
    fig, axes = plt.subplots(1, PER_COLORWAY, figsize=(15, 2.6))
    fig.suptitle(f'{d.parent.name} / {d.name}', x=0.02, ha='left')
    for ax in axes: ax.axis('off')
    for ax, f in zip(axes, files):
        ax.imshow(Image.open(f)); ax.set_title(f.name, fontsize=8)
    plt.show()

In [ ]:
shutil.make_archive('/content/gallery', 'zip', '/content', 'gallery')
from google.colab import files
files.download('/content/gallery.zip')

## Next

Unzip `gallery.zip`, then on GitHub use **Add file → Upload files** and drag the whole `gallery` folder in.
In the app, open **About → Rebuild colorway index**.